In [4]:
import pandas as pd

In [5]:
df=pd.read_csv(r"NLP-Learning\datasets\FakeNews\train.csv.csv") ## using fake news data from NLP

FileNotFoundError: [Errno 2] No such file or directory: 'NLP-Learning\\datasets\\FakeNews\\train.csv.csv'

In [38]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [39]:
# ## get the independent features
# X=df.drop('label', axis = 1)
# y=df['label']


df = df.dropna(subset=['title'])
df = df.reset_index(drop=True)

X = df.drop('label', axis=1)
y = df['label']

messages = X.copy()

In [40]:
## check if dataset is balanced or not
y.value_counts()

,count
label,
0,10387
1,9855


In [41]:
import tensorflow as tf

In [42]:
tf.__version__

'2.20.0'

In [43]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM, Dense, Bidirectional

In [44]:
## vocabulary size
voc_size = 5000

In [45]:
# messages=X.copy()

In [46]:
import nltk
import re
from nltk.corpus import stopwords

In [47]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [48]:
from nltk import PorterStemmer
ps = PorterStemmer()

corpus = []
# messages = messages.dropna(subset=['title'])
# messages = messages.reset_index(drop=True)
for i in range(0, len(messages)):
  # print(i)
  # print(type(messages['title'].iloc[i]))
  review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
  review = review.lower()
  review = review.split()

  review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)

  corpus.append(review)


In [49]:
corpus[:5]

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri']

In [50]:
one_hot_repr = [one_hot(words, voc_size) for words in corpus]
print(one_hot_repr)

[[2310, 2251, 984, 2564, 4055, 3348, 2681, 2412, 2630, 3918], [4870, 3749, 1505, 91, 2456, 2000, 2399], [2922, 2311, 288, 1663], [4127, 2572, 786, 3893, 1584, 4061], [947, 2456, 4198, 585, 511, 3167, 2456, 4342, 613, 3413], [88, 3105, 4004, 4406, 2269, 2552, 3692, 3076, 1411, 633, 2925, 315, 1057, 313, 2399], [17, 17, 1621, 775, 485, 3929, 752, 702, 1990, 1491, 2293, 3283], [4819, 4129, 479, 1431, 278, 2977, 1191, 1750, 4977, 3759, 3624], [4243, 500, 1798, 1631, 2552, 955, 4516, 4956, 2041, 1229, 4977, 3759, 3624], [4345, 3248, 3147, 3426, 1802, 1479, 2552, 328, 4977, 3759, 3624], [4685, 1047, 4106, 1488, 4573, 4210, 3236, 4750, 2552, 1905], [868, 1577, 839, 2318, 1206, 4956, 3987, 1724], [3229, 4985, 4726, 3383, 548, 2403, 2380, 2997, 4587, 3748, 4690], [3893, 1799, 4055, 4210, 2552, 1802], [2381, 2452, 4738, 1483, 4438, 732, 4342, 4485, 2046], [4931, 1746, 1662, 1994, 259, 2397, 2344, 4977, 3759, 3624], [2406, 4298, 1733, 4706, 2522, 4977, 3759, 3624], [4025, 2002, 4282, 1760, 3091, 

In [51]:
## embedding representation

sent_length = 20
embedded_docs = pad_sequences(one_hot_repr, padding='pre', maxlen=sent_length)
embedded_docs

array([[   0,    0,    0, ..., 2412, 2630, 3918],
       [   0,    0,    0, ..., 2456, 2000, 2399],
       [   0,    0,    0, ..., 2311,  288, 1663],
       ...,
       [   0,    0,    0, ..., 4977, 3759, 3624],
       [   0,    0,    0, ..., 3324,  335, 2286],
       [   0,    0,    0, ..., 3765, 2128, 2275]], dtype=int32)

In [52]:
## Creating model
embedding_vector_features = 40
model = Sequential()
model.add(Embedding(voc_size, embedding_vector_features, input_shape=(sent_length,)))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 200)            │       112,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 313,001 (1.19 MB)

 Trainable params: 313,001 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

None


In [53]:
import numpy as np
X_final = np.array(embedded_docs)
y_final = np.array(y)

In [54]:
## train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [55]:
## model training

model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=64)

Epoch 1/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.8941 - loss: 0.2780 - val_accuracy: 0.9144 - val_loss: 0.2058
Epoch 2/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9465 - loss: 0.1411 - val_accuracy: 0.9153 - val_loss: 0.2014
Epoch 3/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9694 - loss: 0.0854 - val_accuracy: 0.9195 - val_loss: 0.2074
Epoch 4/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9835 - loss: 0.0504 - val_accuracy: 0.9223 - val_loss: 0.2495
Epoch 5/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9910 - loss: 0.0305 - val_accuracy: 0.9187 - val_loss: 0.3084
Epoch 6/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9962 - loss: 0.0157 - val_accuracy: 0.9162 - val_loss: 0.3934
Epoch 7/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9973 - loss: 0.0114 - val_accuracy: 0.9162 - val_loss: 0.4254
Epoch 8/10
212/212 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9984 - loss: 0.0071 - val_accu

In [57]:
y_pred = model.predict(X_test)

209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [58]:
y_pred = np.where(y_pred > 0.5, 1, 0)

In [59]:
y_pred

array([[0],
       [0],
       [0],
       ...,
       [1],
       [1],
       [1]])

In [60]:
from sklearn.metrics import confusion_matrix, accuracy_score
confusion_matrix(y_test, y_pred)


array([[3093,  317],
       [ 274, 2996]])

In [61]:
accuracy_score(y_test, y_pred)

0.9115269461077844